## Contexte
Il existe une forte concurrence entre les fournisseurs d’accès Internet. Si un prestataire veut augmenter ses revenus, il a besoin de plus d’abonnés, mais garder les clients existants est plus important que d’en avoir de nouveaux. Les prestataires veulent donc savoir quel client devrait annuler son service. On appelle ça le churn. S’ils savent qui y ira, peut-être pourront-ils les attraper avec des promotions.

In [131]:
project_dir = '/content/drive/MyDrive/project/data_analytics/churn'
from pathlib import Path

%cd $project_dir

/content/drive/MyDrive/project/data_analytics/churn


In [132]:
import pandas as pd

In [133]:
df = pd.read_csv('data/internet_service_churn.csv')
df.columns

Index(['id', 'is_tv_subscriber', 'is_movie_package_subscriber',
       'subscription_age', 'bill_avg', 'reamining_contract',
       'service_failure_count', 'download_avg', 'upload_avg',
       'download_over_limit', 'churn'],
      dtype='object')

In [134]:
data = df.rename(columns={
    "id": "id_client",
    "is_tv_subscriber": "abonnement_tv",
    "is_movie_package_subscriber": "abonnement_cinema",
    "subscription_age": "anciennete_annees",
    "bill_avg": "facture_moyenne",
    "reamining_contract": "engagement_restant_annees",
    "service_failure_count": "nombre_appels_support",
    "download_avg": "telechargement_moyen_go",
    "upload_avg": "envoi_moyen_go",
    "download_over_limit": "nombre_depassements_limite",
    "churn": "resiliation"
}).copy()

| Variable d’origine | Nom en français | Signification |
|---|---|---|
| `id` | `id_client` | Identifiant unique du client |
| `is_tv_subscriber` | `abonnement_tv` | Abonnement à la télévision, avec 1 = oui et 0 = non |
| `is_movie_package_subscriber` | `abonnement_cinema` | Abonnement à l’option cinéma, avec 1 = oui et 0 = non |
| `subscription_age` | `anciennete_annees` | Ancienneté du client en années |
| `bill_avg` | `facture_moyenne` | Facture moyenne sur les trois derniers mois |
| `reamining_contract` | `engagement_restant_annees` | Durée d’engagement restante en années |
| `service_failure_count` | `nombre_appels_support` | Nombre d’appels au support pour un problème de service au cours des trois derniers mois |
| `download_avg` | `telechargement_moyen_go` | Volume moyen de données téléchargées, en Go |
| `upload_avg` | `envoi_moyen_go` | Volume moyen de données envoyées, en Go |
| `download_over_limit` | `nombre_depassements_limite` | Nombre de dépassements de la limite de téléchargement |
| `churn` | `resiliation` | Statut de résiliation, avec 1 = client parti et 0 = client encore abonné |

Pour **engagement_restant_annees**, une valeur manquante correspond à la catégorie « Sans engagement », selon la documentation consultée.

In [135]:
data['id_client'].is_unique

True

In [136]:
data.head(5)

,id_client,abonnement_tv,abonnement_cinema,anciennete_annees,facture_moyenne,engagement_restant_annees,nombre_appels_support,telechargement_moyen_go,envoi_moyen_go,nombre_depassements_limite,resiliation
0,15,1,0,11.95,25,0.14,0,8.4,2.3,0,0
1,18,0,0,8.22,0,NaN,0,0.0,0.0,0,1
2,23,1,0,8.91,16,0.00,0,13.7,0.9,0,1
3,27,0,0,6.87,21,NaN,1,0.0,0.0,0,1
4,34,0,0,6.39,0,NaN,0,0.0,0.0,0,1


In [137]:

# 1. Vérifier la granularité
ids = data["id_client"].dropna()
print(f"Lignes                         : {len(data):,}")
print(f"Identifiants distincts connus  : {ids.nunique():,}")
print(f"Lignes sans identifiant        : {data['id_client'].isna().sum():,}")
print(f"Identifiants présents ≥ 2 fois : {(ids.value_counts() > 1).sum():,}")
print(f"Doublons de ligne exacts       : {data.duplicated().sum():,}")

# 2. Profiler chaque colonne
audit = pd.DataFrame({
    "type": data.dtypes.astype(str),
    "valeurs_distinctes": data.nunique(),
    "valeurs_manquantes": data.isna().sum(),
    "part_manquante_pct": data.isna().mean().mul(100).round(2)
})
display(audit)

# 3. Vérifier le codage du statut et des options d'abonnement
catg_cols = ["abonnement_cinema", "abonnement_tv", "resiliation"]
for col in  catg_cols:
    print(col)
    display(data[col].value_counts(dropna=False , normalize=True).rename("nombre_de_lignes"))

# 4. Repérer les plages de valeurs à documenter
mesures = [cols for cols in data.columns if cols not in catg_cols]
display(data[mesures].describe(include="all").T)

Lignes                         : 72,274
Identifiants distincts connus  : 72,274
Lignes sans identifiant        : 0
Identifiants présents ≥ 2 fois : 0
Doublons de ligne exacts       : 0


,type,valeurs_distinctes,valeurs_manquantes,part_manquante_pct
id_client,int64,72274,0,0.00
abonnement_tv,int64,2,0,0.00
abonnement_cinema,int64,2,0,0.00
anciennete_annees,float64,1110,0,0.00
facture_moyenne,int64,179,0,0.00
engagement_restant_annees,float64,247,21572,29.85
nombre_appels_support,int64,19,0,0.00
telechargement_moyen_go,float64,2856,381,0.53
envoi_moyen_go,float64,802,381,0.53
nombre_depassements_limite,int64,8,0,0.00


abonnement_cinema


,nombre_de_lignes
abonnement_cinema,
0,0.665371
1,0.334629


abonnement_tv


,nombre_de_lignes
abonnement_tv,
1,0.815259
0,0.184741


resiliation


,nombre_de_lignes
resiliation,
1,0.554141
0,0.445859


,count,mean,std,min,25%,50%,75%,max
id_client,72274.0,846318.243462,489102.168744,15.00,422216.50,847784.00,1269562.00,1689744.00
anciennete_annees,72274.0,2.450051,2.034990,-0.02,0.93,1.98,3.30,12.80
facture_moyenne,72274.0,18.942483,13.215386,0.00,13.00,19.00,22.00,406.00
engagement_restant_annees,50702.0,0.716039,0.697102,0.00,0.00,0.57,1.31,2.92
nombre_appels_support,72274.0,0.274234,0.816621,0.00,0.00,0.00,0.00,19.00
telechargement_moyen_go,71893.0,43.689911,63.405963,0.00,6.70,27.80,60.50,4415.20
envoi_moyen_go,71893.0,4.192076,9.818896,0.00,0.50,2.10,4.80,453.30
nombre_depassements_limite,72274.0,0.207613,0.997123,0.00,0.00,0.00,0.00,7.00


In [ ]:
df_clean = data.copy()

In [151]:
# @title
audit_factures = (
    df_clean
    .assign(facture_nulle=df_clean["facture_moyenne"].eq(0))
    .groupby("facture_nulle")
    .agg(
        nombre_clients=("id_client", "size"),
        part_resilies=("resiliation", "mean"),
        anciennete_mediane=("anciennete_annees", "median"),
        anciennetes_invalides=("anciennete_annees", "sum"),
        telechargement_median_go=("telechargement_moyen_go", "median"),
        usages_renseignes=("telechargement_moyen_go", "count")
    )
)

audit_factures["part_resilies_pct"] = (
    audit_factures.pop("part_resilies").mul(100).round(2)
)

display(audit_factures)

,nombre_clients,anciennete_mediane,anciennetes_invalides,telechargement_median_go,usages_renseignes,part_resilies_pct
facture_nulle,,,,,,
False,67101,1.98,166392.69,31.0,66997,52.54
True,5173,1.98,10682.33,0.0,4896,92.65


>Les clients à facture moyenne nulle représentent 7,16 % de la base et présentent une proportion de résiliations de 92,65 %. Leur ancienneté médiane atteint 1,98 an. Ce segment nécessite une vérification du statut de facturation et de la chronologie des données pour préciser la relation entre facture nulle, absence d’usage et résiliation.

In [152]:
# @title
clients_facture_nulle = df_clean.loc[
    df_clean["facture_moyenne"].eq(0)
].copy()

clients_facture_nulle["profil_usage"] = "Téléchargement positif"

clients_facture_nulle.loc[
    clients_facture_nulle["telechargement_moyen_go"].eq(0),
    "profil_usage"
] = "Téléchargement nul"

clients_facture_nulle.loc[
    clients_facture_nulle["telechargement_moyen_go"].isna(),
    "profil_usage"
] = "Téléchargement manquant"

analyse = (
    clients_facture_nulle
    .groupby("profil_usage")
    .agg(
        nombre_clients=("id_client", "size"),
        nombre_resiliations=("resiliation", "sum"),
        part_resilies=("resiliation", "mean"),
        anciennete_mediane=("anciennete_annees", "median")
    )
)

analyse["part_resilies_pct"] = (
    analyse.pop("part_resilies").mul(100).round(2)
)

display(analyse)

,nombre_clients,nombre_resiliations,anciennete_mediane,part_resilies_pct
profil_usage,,,,
Téléchargement manquant,277,0,0.02,0.00
Téléchargement nul,3172,3168,1.98,99.87
Téléchargement positif,1724,1625,2.09,94.26


In [153]:
# @title
df_clean["facture_nulle"] = (
    df_clean["facture_moyenne"].eq(0)
)

df_clean["telechargement_manquant"] = (
    df_clean["telechargement_moyen_go"].isna()
)

df_clean["envoi_manquant"] = (
    df_clean["envoi_moyen_go"].isna()
)

**Traitements des factures nulles**
> Les 5173 factures nulles sont conservées. Leur analyse révèle un groupe de clients très récents, encore abonnés et sans téléchargement renseigné, ainsi que deux groupes très majoritairement résiliés. Les valeurs manquantes d’usage sont conservées et signalées par des indicateurs.

In [146]:
# @title
data.query('anciennete_annees < 0') # retrive all customers who have negative subsc age

# 1 seul client est associe a une anciennete negative , nous allons la remplacer par nan

df_clean['anciennete_annees'] = data['anciennete_annees'].apply(lambda x : pd.NA if x < 0 else x)
df_clean.query('anciennete_annees < 0') # test

,id_client,abonnement_tv,abonnement_cinema,anciennete_annees,facture_moyenne,engagement_restant_annees,nombre_appels_support,telechargement_moyen_go,envoi_moyen_go,nombre_depassements_limite,resiliation


In [140]:
# data.query("envoi_moyen_go.isnull() == True  & telechargement_moyen_go.isnull() == True").describe()

In [187]:

filtre = df_clean['telechargement_moyen_go'].quantile(0.99)
mask = df_clean['telechargement_moyen_go'] >= 1000
mask1 = df_clean['telechargement_moyen_go'] >= filtre
print(df_clean.loc[mask1,:].__len__() , f'dépasse {filtre.round(2)} Go de telechargement et qui correspond aux 3e quartile ')
print(df_clean.loc[mask,:].__len__() , "dépasse 1To de telechargement")


720 dépasse 237.2 Go de telechargement et qui correspond aux 3e quartile 
23 dépasse 1To de telechargement


**Volumes de téléchargement élevés**
> Les valeurs élevées sont conservées, leur niveau étant compatible avec des usages intensifs. Environ 1 % des observations renseignées dépassent 237,2 Go et 23 clients dépassent 1 To. Ces observations seront prises en compte dans la segmentation des usages.

In [141]:
# @title
# !pip -q install duckdb
# import duckdb
# duckdb.query("SELECT * FROM data").to_df()


In [142]:
# data['contrat'] =data['engagement_restant_annees'].map(lambda x : 'sans encagement' if pd.isna(x) == True else 'avec engagement')
# data.head(5)

| Point contrôlé | Décision retenue |
|---|---|
| Identifiants et doublons | Une ligne par client, avec des identifiants uniques et renseignés |
| Ancienneté négative | Remplacement par une valeur manquante, avec conservation du client |
| Factures nulles | Conservation, avec un indicateur `facture_nulle` |
| Usages manquants | Conservation des valeurs manquantes et ajout d’indicateurs |
| Volumes de téléchargement élevés | Conservation comme usages intensifs plausibles |
| Engagement | Ajout de ta variable `contrat` pour faciliter les comparaisons |

In [192]:
df_clean.to_csv(project_dir+'/data/data_cleaned.csv' , index=False)